# データの前処理

In [ ]:
# 処理状況の確認
# --- ご自身のパスに合わせてください．
# 現在、結果が追記されている出力ファイルのパス
output_file_path = './opensubtitles_ja_en_all_basemodel_translation.csv'

# --- 設定ここまで -./

import os

try:
    if os.path.exists(output_file_path):
        # ファイルを開き、行数を数える (メモリ効率の良い方法)
        with open(output_file_path, 'r', encoding='utf-8') as f:
            line_count = sum(1 for line in f)

        # ヘッダー行を引いた数が、処理済みのデータ行数
        processed_rows = line_count - 1
        print(f"現在、{processed_rows} 行目まで処理が完了しています。")

    else:
        print("出力ファイルはまだ作成されていません。処理は0行目です。")

except Exception as e:
    print(f"ファイルの読み込み中にエラーが発生しました: {e}")

In [ ]:
# 文字化けを確認する

import pandas as pd

# --- ユーザー設定項目 ---
file_path = './opensubtitles_ja_en_all_basemodel_translation.csv'
column_to_check = 'baseline_translation'
# --- 設定ここまで ---

try:
    print(f"ファイル ({file_path}) を確認中...")
    df = pd.read_csv(file_path)

    # 確認1：空欄（nullまたは空文字列）のチェック
    empty_count = df[column_to_check].isnull().sum()
    blank_string_count = (df[column_to_check] == '').sum()
    total_empty = empty_count + blank_string_count

    if total_empty > 0:
        print(f"\n[警告] {column_to_check} 列に {total_empty} 件の空欄が見つかりました。")
    else:
        print(f"\n[OK] {column_to_check} 列に空欄はありませんでした。")

    # 確認2：文字化けの簡易チェック（豆腐"□"や""の存在を確認）
    garbled_chars = ['□', '']
    garbled_count = 0
    for char in garbled_chars:
        # 文字化けの可能性がある文字を含む行をカウント
        count = df[df[column_to_check].str.contains(char, na=False)].shape[0]
        if count > 0:
            print(f"\n[警告] 文字化けの可能性（'{char}'）が {count} 件見つかりました。")
            garbled_count += count

    if garbled_count == 0:
        print(f"\n[OK] 簡単な文字化けチェックでは問題は見つかりませんでした。")

    print("\n確認完了。")

except FileNotFoundError:
    print(f"エラー: ファイルが見つかりません。パスを確認してください: {file_path}")
except Exception as e:
    print(f"エラーが発生しました: {e}")

In [ ]:
# クリーニングする

import pandas as pd

# --- ご自身のパスに合わせてください．

# 1. 入力するCSVファイルのパス (翻訳結果を含むファイル)
input_file_path ='./opensubtitles_ja_en_all_basemodel_translation.csv'

# 2. クリーンなデータを保存する新しい出力ファイルパス
output_cleaned_path ='./opensubtitles_ja_en_all_basemodel_translation_cleaned.csv'

# 3. チェック対象の列
column_to_check = 'baseline_translation'

# --- 設定ここまで ---

try:
    print(f"ファイル ({input_file_path}) を読み込み中...")
    df = pd.read_csv(input_file_path)
    initial_row_count = len(df)
    print(f"読み込み完了。合計 {initial_row_count} 行のデータがあります。")

    # --- 空欄行の削除 ---
    # 条件1: Null値である
    is_null = df[column_to_check].isnull()

    # 条件2: 空文字列である
    # .astype(str) を使って、非文字列データが混在していてもエラーが出ないようにする
    is_empty_string = (df[column_to_check].astype(str) == '')

    # Nullまたは空文字列の行を特定
    rows_to_drop = is_null | is_empty_string

    # 削除対象の行数をカウント
    drop_count = rows_to_drop.sum()

    if drop_count > 0:
        print(f"\n{drop_count} 件の空欄（Nullまたは空文字列）の行を削除します...")
        # 条件に合わない行だけを残す（~は否定を意味する）
        df_cleaned = df[~rows_to_drop].copy()
        print(" -> 削除完了。")
    else:
        print("\n削除対象の空欄行はありませんでした。")
        df_cleaned = df.copy()


    # --- ファイルに保存 ---
    print(f"\nクリーンなデータを '{output_cleaned_path}' として保存します...")
    df_cleaned.to_csv(output_cleaned_path, index=False, encoding='utf-8')
    print(" -> 保存完了。")

    print("\n--- 処理完了 ✨ ---")
    print(f"最終的なデータ: {len(df_cleaned)} 行")

except FileNotFoundError:
    print(f"エラー: ファイルが見つかりません。パスを確認してください: {input_file_path}")
except Exception as e:
    print(f"エラーが発生しました: {e}")